# Resume2Roles

## Demonstration - Includes user interface
Render using "voila [path]" in the command line.

Requires the Voila python tool to be installed.


---



## Authors:
* Ethan Eckmann
* Eddie Lin
* Henry Wei
* Neha Indolikar



---



## Description:
This project is a Natural Language Processing system that parses resumes to extract relevant information such as skills, experience, education, etc. to return a ranked list of the best matching job roles sourced from Linkedin job postings for a given candidate. Job roles are ranked by the Cosine Similarity scores between the resume and job postings in the dataset to quantify how well a candidate matches a given job role.

Multiple text vectorization methods are provided:

TF-IDF, Word2Vec, GloVe, and Sentence BERT


In [1]:
%%capture
import importlib.util
import os
import subprocess
import sys

def install_and_import(package_name):
    """
    Checks if a package is installed. If not, installs it and then imports it.
    """
    # Check if the package is already installed
    spec = importlib.util.find_spec(package_name)

    if spec is None:
        print(f"{package_name} not found. Installing...")
        try:
            # Use sys.executable to ensure it's installed in the correct environment
            subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
            print(f"Successfully installed {package_name}")
        except subprocess.CalledProcessError as e:
            print(f"Failed to install {package_name}: {e}")
            return None
    else:
        print(f"{package_name} is already installed.")

    # Import the package programmatically
    return importlib.import_module(package_name)

install_and_import('bert_score')
install_and_import('kagglehub')
install_and_import('nltk')
install_and_import('sklearn')
install_and_import('wordcloud')
install_and_import('gensim')
install_and_import('sentence_transformers')
install_and_import('torch')
install_and_import('zipfile')

# libraries
import pandas as pd
import kagglehub
import shutil
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

import nltk
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
import re
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
from gensim.models import Word2Vec

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers.util import cos_sim as cosine_similarity_sbert

import ipywidgets as widgets
from IPython.display import display

In [2]:
# Function to handle dataset download and aliasing
def download_and_alias_dataset(dataset_id, target_alias_path):
    path = kagglehub.dataset_download(dataset_id)
    # Check if the target alias path already exists
    if not os.path.exists(target_alias_path):
        # kagglehub.dataset_download usually returns the path to a directory.
        # shutil.move attempts os.rename first, which fails across different filesystems (e.g., /kaggle/input to /content).
        # It then falls back to copying the contents and trying to delete the original source.
        # The '/kaggle/input' filesystem is read-only, so the deletion step fails with OSError 30.
        # To fix this, we should directly copy the directory using shutil.copytree,
        # which doesn't attempt to delete the source.
        shutil.copytree(path, target_alias_path)


In [3]:
# Preprocess function
def preprocess(text):

  if not isinstance(text, str):
    return ""

  # Map special tech skills to text equivalents so they aren't destroyed by punctuation removal
  text = text.replace('c++', 'cplusplus')
  text = text.replace('c#', 'csharp')
  text = text.replace('.net', 'dotnet')
  text = text.replace('f#', 'fsharp')

  # Safely remove all special characters (reverting to your original safe regex)
  text = re.sub(r'[^a-z0-9\s]', '', text)



  tokens = text.split()
  tokens = [token for token in tokens if token not in stop_words]
  tokens = [lemmatizer.lemmatize(t) for t in tokens]
  return " ".join(tokens)

In [4]:
# loading non-preprocessed datasets
# if datasets are not in the local directory, load them
import os

data_directory_path = 'datasets'

# Joining paths ensures it works on both Windows and Mac/Linux
file_path_1 = os.path.join(data_directory_path, 'postings.csv')
file_path_2 = os.path.join(data_directory_path, 'resumes.csv')

if not os.path.isfile(file_path_1) or not os.path.isfile(file_path_2):
    # Download and preprocess datasets
    print("Missing data files. Downloading from Kaggle")
    download_and_alias_dataset("arshkon/linkedin-job-postings", '/content/arshkon_linkedin-job-postings')
    download_and_alias_dataset("saugataroyarghya/resume-dataset", '/content/saugataroyarghya_resume-dataset')

    job_df_processed = pd.read_csv('/content/arshkon_linkedin-job-postings/postings.csv')
    resume_df = pd.read_csv('/content/saugataroyarghya_resume-dataset/resume_data.csv')

    # fix error (hidden character (a Byte Order Mark or BOM))
    resume_df.rename(columns={'\ufeffjob_position_name': 'job_position_name'}, inplace=True)

    # select columns to use
    job_df = job_df_processed[['description', 'title']]
    resume_df['combined'] = resume_df['career_objective'] + resume_df['skills'] + resume_df['major_field_of_studies']
    resume_df = resume_df[['combined', 'job_position_name']]

    # Create "datasets" directory if it doesn't exist
    if not os.path.exists(data_directory_path):
        os.makedirs(data_directory_path)

    # save datasets
    job_df_processed.to_csv(file_path_1, index=False)
    resume_df.to_csv(file_path_2, index=False)

else:
    # load datasets
    print("Loading datasets from local")
    job_df = pd.read_csv(file_path_1)
    resume_df = pd.read_csv(file_path_2)

Loading datasets from local


In [5]:
# loading preprocessed datasets
# if datasets are not in the local directory, load them
import os
import numpy as np

data_directory_path = 'datasets'

# Joining paths ensures it works on both Windows and Mac/Linux
file_path_3 = os.path.join(data_directory_path, 'postings_processed.csv')
file_path_4 = os.path.join(data_directory_path, 'resumes_processed.csv')

if not os.path.isfile(file_path_3) or not os.path.isfile(file_path_4):
    # Preprocess datasets
    print("Preprocessing datasets")
   
    # apply preprocessing on the text blob of the job applications
    job_df_processed = job_df.copy()
    job_df_processed['description'] = job_df_processed['description'].apply(preprocess)

    # apply preprocessing on the text blob of the resumes
    resume_df_processed = resume_df.copy()
    resume_df_processed['combined'] = resume_df_processed['combined'].apply(preprocess)

    # save preprocessed datasets
    job_df_processed.to_csv(file_path_3, index=False)
    resume_df_processed.to_csv(file_path_4, index=False)

else:
    # load preprocessed datasets
    print("Loading preprocessed datasets from local")
    job_df_processed = pd.read_csv(file_path_3)
    resume_df_processed = pd.read_csv(file_path_4)

Loading preprocessed datasets from local


In [6]:
# Remove rows with NaN
import numpy as np

# Convert literal empty strings ('') to NaN
resume_df['combined'] = resume_df['combined'].replace('', np.nan)
job_df['description'] = job_df['description'].replace('', np.nan)

# Drop any rows that have NaN in the text columns
resume_df.dropna(subset=['combined'], inplace=True)
job_df.dropna(subset=['description'], inplace=True)

print(f"Rows remaining in resume_df: {len(resume_df)}")
print(f"Rows remaining in job_df: {len(job_df)}")

Rows remaining in resume_df: 4712
Rows remaining in job_df: 123842


In [7]:
# Remove rows with NaN
import numpy as np

# Convert literal empty strings ('') to NaN
resume_df_processed['combined'] = resume_df_processed['combined'].replace('', np.nan)
job_df_processed['description'] = job_df_processed['description'].replace('', np.nan)

# Drop any rows that have NaN in the text columns
resume_df_processed.dropna(subset=['combined'], inplace=True)
job_df_processed.dropna(subset=['description'], inplace=True)

print(f"Rows remaining in resume_df_processed: {len(resume_df_processed)}")
print(f"Rows remaining in job_df_processed: {len(job_df_processed)}")

Rows remaining in resume_df_processed: 4712
Rows remaining in job_df_processed: 123839


In [8]:
# Tokenize for Word2Vec and GloVe
resume_tokens_word2vec = resume_df_processed['combined'].fillna("").str.lower().str.split().tolist()
job_tokens_word2vec =  job_df_processed['description'].fillna("").str.lower().str.split().tolist()

job_tokens_glove = job_df_processed['description'].fillna("").str.lower().str.split().tolist()

In [9]:
# TFIDF vectorizer 
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(
    ngram_range = (1,2), #include both unigrams and bigram
    max_features=10000, # Keeps only the top 10k most frequent n-grams
    min_df=2            # Ignores words that only appear in one single document
)

In [10]:
# Word2Vec model
try:
    model_word2vec = Word2Vec.load("word2vec.model")
except:
    print('training word2vec model')
    all_text_tokenized = resume_tokens_word2vec + job_tokens_word2vec

    model_word2vec = Word2Vec(sentences=all_text_tokenized, vector_size=300, window=5, min_count=1, workers=4)

    # Save model
    model_word2vec.save("word2vec.model")

In [11]:
# Word2Vec vectorizer
def get_document_vector_word2vec(tokens, model):
    # Filter out words that are not in the model's vocabulary
    feature_vectors = [model.wv[word] for word in tokens if word in model.wv]
    
    # If the document is empty or no words match the vocabulary, return a zero vector
    if not feature_vectors:
        return np.zeros(model.vector_size)
    
    # Calculate the average of all word vectors
    return np.mean(feature_vectors, axis=0)

In [12]:
# GloVe vectorizer
class GloveVectorizer:
    def __init__(self, loaded_model):
        """
        Parameters:
        - model_name: name of the pre-trained GloVe model
        """
        self.model = loaded_model
        self.embedding_dim = self.model.vector_size

    def transform_word(self, word):
        """
        Return embedding for a single word (for visualization/debugging).
        """
        return self.model.get(word, np.zeros(self.embedding_dim))

    def transform(self, documents_tokens):
        """
        Convert tokenized documents into sentence vectors
        by averaging word embeddings.
        """
        vectors = []
        for tokens in documents_tokens:
            token_vectors = [
                self.model[token] for token in tokens if token in self.model
            ]
            if len(token_vectors) == 0:
                vectors.append(np.zeros(self.embedding_dim))
            else:
                vectors.append(np.mean(token_vectors, axis=0))
        return np.array(vectors)
    
glove_model_200 = api.load("glove-wiki-gigaword-200")
glove_vectorizer = GloveVectorizer(glove_model_200)

In [13]:
import os
import zipfile
import shutil
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
import torch
from transformers import logging as transformers_logging

# This silences the TQDM progress bars for downloading/loading weights
transformers_logging.set_verbosity_error()

# Automatically detect if a GPU is available, otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Check if the zip file exists and unzip it if the directory doesn't exist yet
if os.path.exists("sbert.model.zip") and not os.path.exists("sbert.model"):
    print("Found zipped model, extracting...")
    with zipfile.ZipFile("sbert.model.zip", 'r') as zip_ref:
        zip_ref.extractall("sbert.model")

# Check if the model is already saved locally
if os.path.exists("sbert.model"):
    print("Loading SBERT model from local directory...")
    model_sentbert = SentenceTransformer("sbert.model", device=device)
else:
    print("Downloading SBERT model from Hugging Face...")
    model_sentbert = SentenceTransformer("sentence-transformers/all-MiniLM-L12-v2", device=device)
    model_sentbert.save("sbert.model")
    shutil.make_archive("sbert.model", 'zip', "sbert.model")

# Encode the preprocessed texts with a larger batch size for faster GPU processing
batch_size = 256

Loading SBERT model from local directory...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [14]:
print(f"The maximum sequence length for the sentence BERT model is: {model_sentbert.max_seq_length}")

The maximum sequence length for the sentence BERT model is: 128


In [15]:
# load previously saved vectors

job_vectorized_tfidf = tfidf_vectorizer.fit_transform(job_df_processed['description'])

try:
    job_vectorized_word2vec = np.load('word2vec_job_vectors.npy',allow_pickle=True)
except:
    print('Vectorizing job_vectorized_word2vec')
    job_vectorized_word2vec = np.array([get_document_vector_word2vec(it, model_word2vec) for it in job_tokens_word2vec])
    np.save('word2vec_job_vectors.npy', job_vectorized_word2vec)

try:
    job_vectorized_glove = np.load('glove_job_vectors.npy',allow_pickle=True)
except:
    print('Vectorizing job_vectorized_glove')
    job_vectorized_glove = glove_vectorizer.transform(job_tokens_glove)
    np.save('glove_job_vectors.npy', job_vectorized_glove)

try:
    job_embeddings_sentbert = np.load('sentbert_job_embeddings.npy',allow_pickle=True)
except:
    print('Embedding job_embeddings_sentbert')
    job_embeddings_sentbert = model_sentbert.encode(job_df['description'].tolist(), batch_size=batch_size, show_progress_bar=True)
    np.save('sentbert_job_embeddings.npy', job_embeddings_sentbert)



In [16]:
def return_top_job_for_text_tfidf(resume_text, top, job_vectors, job_df, tfidf_vectorizer):
  '''
  find the top relevant job postings based on the resume
  Parameters
  ----------
  resume_text: string
    the text blob of a resume
  top: int
    the top number of matches to return

  Return
  ------
  results: list
    a list of dictionaries that contain the information about the top
    matches
  '''
  resume_vector = tfidf_vectorizer.transform([resume_text])

  # Compute Similarity using the single document vector
  similarity_scores = cosine_similarity(resume_vector, job_vectors)
  scores = similarity_scores[0]
  top_index = np.argsort(scores)[::-1] 

  results = []
  rank = 1
  seen_titles = set()

  # Iterate through all potential matches until you fill your 'top' requirement
  for job_index in top_index:
      title = job_df.iloc[job_index]['title']
      if title not in seen_titles:
          results.append({
              'Rank': rank,
              'Job Title': title,
              'Match Score': round(scores[job_index], 2)
          })
          seen_titles.add(title)
          rank += 1
      
      # Stop once the results list reaches the desired 'top' count (e.g., 10)
      if len(results) >= top:
          break
  return results

In [17]:
def return_top_job_for_text_word2vec(resume_text, top, job_vectors, job_df, model):
  '''
  find the top relevant job postings based on the resume
  Parameters
  ----------
  resume_text: string
    the text blob of a resume
  top: int
    the top number of matches to return

  Return
  ------
  results: list
    a list of dictionaries that contain the information about the top
    matches
  '''
  resume_tokens = preprocess(resume_text).split() # Preprocess first!
  resume_vector = get_document_vector_word2vec(resume_tokens, model).reshape(1, -1)

  # Compute Similarity using the single document vector
  similarity_scores = cosine_similarity(resume_vector, job_vectors)
  scores = similarity_scores[0]
  top_index = np.argsort(scores)[::-1] 

  results = []
  rank = 1
  seen_titles = set()

  # Iterate through all potential matches until you fill your 'top' requirement
  for job_index in top_index:
      title = job_df.iloc[job_index]['title']
      if title not in seen_titles:
          results.append({
              'Rank': rank,
              'Job Title': title,
              'Match Score': round(scores[job_index], 2)
          })
          seen_titles.add(title)
          rank += 1
      
      # Stop once the results list reaches the desired 'top' count (e.g., 10)
      if len(results) >= top:
          break
  return results

In [18]:
def return_top_job_for_text_glove(resume_text, top, job_vectors, job_df, glove_vectorizer):
  '''
  find the top relevant job postings based on the resume
  Parameters
  ----------
  resume_text: string
    the text blob of a resume
  top: int
    the top number of matches to return

  Return
  ------
  results: list
    a list of dictionaries that contain the information about the top
    matches
  '''

  resume_tokens = preprocess(resume_text).lower().split() 
  resume_vector = glove_vectorizer.transform([resume_tokens])
  # Compute Similarity using the single document vector
  similarity_scores = cosine_similarity(resume_vector, job_vectors)
  scores = similarity_scores[0]
  top_index = np.argsort(scores)[::-1] 

  results = []
  rank = 1
  seen_titles = set()

  # Iterate through all potential matches until you fill your 'top' requirement
  for job_index in top_index:
      title = job_df.iloc[job_index]['title']
      if title not in seen_titles:
          results.append({
              'Rank': rank,
              'Job Title': title,
              'Match Score': round(scores[job_index], 2)
          })
          seen_titles.add(title)
          rank += 1
      
      # Stop once the results list reaches the desired 'top' count (e.g., 10)
      if len(results) >= top:
          break
  return results

In [19]:
def return_top_job_for_text_sbert(resume_text, top, job_embeddings, job_df, sbert_model, batch_size):
  '''
  find the top relevant job postings based on the resume
  Parameters
  ----------
  resume_text: string
    the text blob of a resume
  top: int
    the top number of matches to return

  Return
  ------
  results: list
    a list of dictionaries that contain the information about the top
    matches
  '''
  resume_embeddings = sbert_model.encode(resume_text, batch_size=batch_size, show_progress_bar=False)

  # Compute Similarity using the single document vector
  similarity_scores = cosine_similarity_sbert(resume_embeddings, job_embeddings)
  scores = similarity_scores[0].cpu().numpy()
  # scores = similarity_scores[0]
  #  Get all indices sorted by score (remove [:top])
  top_index = np.argsort(scores)[::-1] 

  results = []
  rank = 1
  seen_titles = set()

  # Iterate through all potential matches until you fill your 'top' requirement
  for job_index in top_index:
      title = job_df.iloc[job_index]['title']
      if title not in seen_titles:
          results.append({
              'Rank': rank,
              'Job Title': title,
              'Match Score': round(scores[job_index], 2)
          })
          seen_titles.add(title)
          rank += 1
      
      # Stop once the results list reaches the desired 'top' count (e.g., 10)
      if len(results) >= top:
          break
  return results

In [20]:
# resume_text_1 = """ 
# Alex Taylor San Francisco, CA | (555) 123-4567 | alex.taylor@email.com | linkedin.com/in/alextaylor | github.com/alextaylor
# Professional Summary Results-driven Software Engineer with 4+ years of experience designing and developing scalable web applications. Proficient in full-stack development with a strong focus on backend architecture, system optimization, and writing clean, maintainable code.
# Technical Skills
# Languages: Python, JavaScript (ES6+), Java, SQL
# Frameworks: React, Node.js, Django, Spring Boot
# Tools & Cloud: Git, Docker, Kubernetes, AWS (EC2, S3, RDS), CI/CD pipelines
# Professional Experience Software Engineer | TechNova Solutions June 2022 – Present
# Developed and deployed 5+ RESTful APIs using Node.js and Express, improving data retrieval speeds by 30%.
# Collaborated with cross-functional teams to migrate legacy monolithic architecture to microservices using Docker and AWS.
# Reduced application downtime by 15% through the implementation of automated testing and robust CI/CD pipelines.
# Junior Developer | CodeCrafters Inc. August 2020 – May 2022
# Built responsive frontend components using React, increasing mobile user engagement by 25%.
# Optimized database queries in PostgreSQL, reducing load times for the primary dashboard by 2 seconds.
# Education Bachelor of Science in Computer Science University of California, Berkeley | Graduated: May 2020
# """

# resume_text_2 = """ 
# Alex Taylor New York, NY | (555) 987-6543 | alex.taylor@email.com | linkedin.com/in/alextaylor
# Professional Summary Creative and data-driven Marketing Manager with 6 years of experience leading successful digital campaigns, brand positioning, and content strategies. Proven track record of increasing brand awareness, driving lead generation, and managing multi-channel marketing budgets.
# Core Competencies
# Digital Marketing & SEO/SEM
# Campaign Strategy & Execution
# Social Media Management
# Data Analytics (Google Analytics, HubSpot)
# Content Creation & Copywriting
# Professional Experience Marketing Manager | Elevate Brands March 2021 – Present
# Managed a $500K annual marketing budget, allocating funds across paid social, search, and influencer partnerships to achieve a 250% ROI.
# Spearheaded the launch of a new product line, resulting in $1.2M in sales within the first quarter.
# Grew organic social media following by 150% across Instagram and LinkedIn over 18 months through targeted content strategies.
# Marketing Specialist | Visionary Media July 2018 – February 2021
# Executed email marketing campaigns to a subscriber list of 50K+, maintaining an average open rate of 28%.
# Wrote and edited SEO-optimized blog posts, increasing organic website traffic by 40% year-over-year.
# Education Bachelor of Arts in Communications and Marketing New York University | Graduated: May 2018
# """

# resume_text_3 = """ 
# Alex Taylor, BSN, RN Chicago, IL | (555) 456-7890 | alex.taylor@email.com | linkedin.com/in/alextaylor
# Professional Summary Compassionate and highly skilled Registered Nurse with 5+ years of clinical experience in high-paced medical-surgical and telemetry units. Dedicated to providing patient-centered care, educating patients and families, and collaborating with interdisciplinary healthcare teams to improve patient outcomes.
# Certifications & Skills
# Registered Nurse (RN) – State of Illinois (Active)
# Basic Life Support (BLS) & Advanced Cardiovascular Life Support (ACLS) Certified
# Proficient in Electronic Health Records (Epic, Cerner)
# Patient Assessment & Triage
# Medication Administration & IV Therapy
# Clinical Experience Registered Nurse - Med/Surg Unit | City General Hospital September 2020 – Present
# Manage comprehensive care for up to 6 acute care patients per shift, specializing in post-operative recovery and chronic disease management.
# Communicate effectively with physicians, therapists, and pharmacists to develop and update individualized patient care plans.
# Achieved a 98% positive patient satisfaction rating for bedside manner and clear communication.
# Staff Nurse | Oakridge Medical Center June 2018 – August 2020
# Administered medications, monitored vital signs, and recognized early symptoms of patient deterioration, taking immediate corrective action.
# Mentored 4 newly graduated nurses, providing hands-on training in hospital protocols and clinical documentation.
# Education Bachelor of Science in Nursing (BSN) University of Illinois Chicago | Graduated: May 2018
# """

# UI

In [21]:
# Use Textarea for resumes (handles line breaks/size better)
text_area = widgets.Textarea(
    value='',
    placeholder='Paste resume text here',
    description='Resume:',
    layout={'width': '90%', 'height': '200px'}
)

int_input = widgets.BoundedIntText(
    value=3,
    min=1,          # Minimum value allowed
    max=300,        # Optional: Maximum value allowed
    step=1,         # Ensures only whole numbers
    description='Number of results:',
    style={'description_width': 'initial'},
    layout={'width': '300px'}
)

# Toggle vectorization options
toggle_group = widgets.ToggleButtons(
    options=['TFIDF', 'Word2Vec', 'GloVe embeddings', 'Sentence BERT'],
    description='Choose vectorization method:',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltips=['TFIDF', 'Word2Vec', 'GloVe embeddings', 'Sentence BERT'],
)


#  Add a Button to trigger the search
search_button = widgets.Button(
    description='Find Matches',
    button_style='success', 
    icon='search'
)

# Create an Output widget to display results cleanly
output_window = widgets.Output()

def run_analysis(b):
    # 'b' is the button object, passed automatically by on_click
    resume_text = text_area.value
    num_results = int_input.value
    method = toggle_group.value
    
    with output_window:
        output_window.clear_output() # Clears previous results so they don't stack
        if not resume_text.strip():
            print("Please enter some text first.")
            return
                    
        try:
            if method == 'TFIDF':
                results = return_top_job_for_text_tfidf(resume_text, num_results, job_vectorized_tfidf, job_df_processed, tfidf_vectorizer)
            elif method == 'Word2Vec':  
                results = return_top_job_for_text_word2vec(resume_text, num_results, job_vectorized_word2vec, job_df_processed, model_word2vec)
            elif method == 'GloVe embeddings':  
                results = return_top_job_for_text_glove(resume_text, num_results, job_vectorized_glove, job_df_processed, glove_vectorizer)
            elif method == 'Sentence BERT':  
                results = return_top_job_for_text_sbert(resume_text, num_results, job_embeddings_sentbert, job_df, model_sentbert, batch_size=256)

        
            
            print(f"--- Top Job Matches ---")
            for job in results:
                print(f"Score: {job['Match Score']:.4f} | Title: {job['Job Title']}")
        except Exception as e:
            print(f"Error during processing: {e}")

# Link the button to the function
search_button.on_click(run_analysis)
# Display everything
ui = widgets.VBox([text_area, int_input, toggle_group, search_button, output_window])
display(ui)